# Attention From Scratch — Visual Walkthrough

This notebook runs a small toy example through `MultiHeadAttention` and visualizes
the attention weight matrix as a heatmap.

The goal: see *which tokens attend to which*, and build an intuition for what
the attention weights actually mean.

In [ ]:
import sys
import os
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Add src/ to the path so we can import our module
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from attention import MultiHeadAttention

## 1. Set up a toy example

We'll use a sequence of 6 tokens with a small embedding size.
The tokens are meaningless random vectors — we're not training anything,
just watching what the (randomly initialized) model does.

In a real model, these vectors would be learned word embeddings.

In [ ]:
torch.manual_seed(42)  # fixed seed so results are reproducible

d_model   = 16
num_heads = 2
seq_len   = 6
batch_size = 1

# Fake token labels — pretend this is the sentence "the cat sat on the mat"
tokens = ["the", "cat", "sat", "on", "the", "mat"]

# Random input: one vector per token
X = torch.randn(batch_size, seq_len, d_model)

print(f"Input shape: {X.shape}  →  (batch={batch_size}, seq_len={seq_len}, d_model={d_model})")

## 2. Run through MultiHeadAttention

In self-attention, the same sequence is used as Query, Key, *and* Value.
Every token asks "how much should I attend to every other token?"

In [ ]:
model = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
model.eval()  # turn off dropout / training behaviour (not used here, but good habit)

with torch.no_grad():  # we're not training, so don't track gradients
    output, attention_weights = model(X, X, X)

print(f"Output shape:           {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print()
print("attention_weights shape is (batch, num_heads, seq_len, seq_len)")
print("  → one [seq_len × seq_len] matrix per head")
print("  → row i = how token i distributed its attention across all tokens")

## 3. Visualize the attention weights

Each cell `[i, j]` shows how much token `i` (row) attended to token `j` (column).
Brighter = more attention. Each row sums to 1.

In [ ]:
def plot_attention(weights, tokens, title="Attention weights"):
    """
    Plot a single [seq_len x seq_len] attention matrix as a heatmap.
    weights: 2D tensor of shape (seq_len, seq_len)
    """
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(weights.numpy(), cmap="Blues", vmin=0, vmax=1)

    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=11)
    ax.set_yticklabels(tokens, fontsize=11)

    ax.set_xlabel("Key (token being attended to)", fontsize=10)
    ax.set_ylabel("Query (token doing the attending)", fontsize=10)
    ax.set_title(title, fontsize=13, pad=12)

    # Print the value inside each cell
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            ax.text(j, i, f"{weights[i, j]:.2f}",
                    ha="center", va="center", fontsize=8,
                    color="white" if weights[i, j] > 0.5 else "black")

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


# attention_weights shape: (batch=1, num_heads=2, seq_len, seq_len)
# Squeeze out the batch dimension, then plot each head separately
weights = attention_weights.squeeze(0)  # → (num_heads, seq_len, seq_len)

for head_idx in range(num_heads):
    plot_attention(
        weights[head_idx],
        tokens,
        title=f"Head {head_idx + 1} — attention weights"
    )

## 4. Check that every row sums to 1

This is the softmax guarantee. Every token distributes exactly 100% of its
attention across the sequence. Print the row sums to confirm.

In [ ]:
for head_idx in range(num_heads):
    row_sums = weights[head_idx].sum(dim=-1)
    print(f"Head {head_idx + 1} row sums: {row_sums.tolist()}")

## 5. Now apply a causal mask

In a decoder (like GPT), token `i` is only allowed to attend to tokens `0..i`.
It cannot see the future.

We create a boolean mask where `True` means "block this position".
The upper triangle (future positions) is set to `True`.

In [ ]:
# torch.ones upper triangle: position [i,j] is True if j > i
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
causal_mask = causal_mask.unsqueeze(0)  # add batch dim → (1, seq_len, seq_len)

print("Causal mask (True = blocked):")
print(causal_mask.squeeze(0).int().numpy())

In [ ]:
with torch.no_grad():
    _, masked_weights = model(X, X, X, mask=causal_mask)

masked_weights = masked_weights.squeeze(0)  # → (num_heads, seq_len, seq_len)

for head_idx in range(num_heads):
    plot_attention(
        masked_weights[head_idx],
        tokens,
        title=f"Head {head_idx + 1} — causal mask applied"
    )

The upper triangle should be all zeros — the future is completely blocked.
Token `i` can only see itself and tokens that came before it.

This is exactly how autoregressive models like GPT generate text:
each token is predicted using only the tokens to its left.